In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQ_C.xpt"
csv_file = DATA_DIR / "SMQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 78)
print("NHANES 2003-2004 SMQ_C XPT -> CSV")
print("=" * 78)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED COLUMNS
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMQ020",
    "SMD030",
    "SMQ040",
    "SMQ050Q",
    "SMQ050U",
    "SMD055",
    "SMD057",
    "SMD070",
    "SMD075",
    "SMQ077",
    "SMD641",
    "SMD650",
    "SMD093",
    "SMDUPCA",
    "SMD100BR",
    "SMD100FL",
    "SMD100MN",
    "SMD100LN",
    "SMD100TR",
    "SMD100NI",
    "SMD100CO",
    "SMQ120",
    "SMD130",
    "SMQ140",
    "SMQ143",
    "SMQ145",
    "SMQ150",
    "SMD160",
    "SMQ170",
    "SMQ173",
    "SMQ175",
    "SMQ180",
    "SMD190",
    "SMQ200",
    "SMD203",
    "SMQ205",
    "SMQ210",
    "SMD220",
    "SMQ230",
    "SMD233",
    "SMD235"
]


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "SMQ_C column names/order differ from expected."
    )

if len(df) != 5041:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )

if len(df.columns) != 42:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )

print(
    "PASS: SMQ_C structure = 5,041 rows x 42 columns."
)


# ============================================================
# 4. FIELD TYPES
# ============================================================

#
# Character variables:
#
# SMDUPCA  = 12-character UPC / special text such as
#            * NO MATCH *
#
# SMD100BR = Cigarette brand/sub-brand
#

CHARACTER_COLUMNS = [
    "SMDUPCA",
    "SMD100BR"
]


#
# These are product measurements.
#
# IMPORTANT:
# DO NOT .round()
# DO NOT .astype(int)
# DO NOT re-calculate
#
# SMD100NI contains genuine decimals such as:
#
# 0.5
# 0.8
# 1.3
#
# They stay AS IS.
#

MEASUREMENT_COLUMNS = [
    "SMD100TR",
    "SMD100NI",
    "SMD100CO"
]


#
# All remaining numeric fields are questionnaire
# codes/counts/ages.
#

CODE_COLUMNS = [
    col
    for col in df.columns
    if col not in (
        ["SEQN"]
        + CHARACTER_COLUMNS
        + MEASUREMENT_COLUMNS
    )
]


# ============================================================
# 5. KNOWN PANDAS XPORT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

true_zero_count = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{true_zero_count:,}"
)

print(
    "Tiny-value occurrences:",
    f"{int(tiny_counts.sum()):,}"
)

print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. CDC-VALIDATED ZERO VARIABLES
# ============================================================

ZERO_ALLOWED_COLUMNS = [
    "SMD030",
    "SMD641",
    "SMD100FL",
    "SMD100MN",
    "SMD130",
    "SMD160",
    "SMD190",
    "SMD220"
]


EXPECTED_TINY_COUNTS = {

    "SMD030": 165,

    # CDC permits SMD641 range 0-30.
    # 12 zeros are found in THIS source XPT.
    "SMD641": 12,

    "SMD100FL": 27,
    "SMD100MN": 732,

    "SMD130": 215,
    "SMD160": 420,
    "SMD190": 38,
    "SMD220": 82
}


# ============================================================
# 7. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)

approved_tiny_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected = (
    actual_tiny_columns
    -
    approved_tiny_columns
)


if unexpected:

    raise ValueError(
        "STOP: Tiny values found in unexpected variables:\n"
        + str(sorted(unexpected))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts.get(
            col,
            0
        )
    )

    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count} tiny values, "
            f"found {actual_count}."
        )


if int(tiny_counts.sum()) != 1691:

    raise ValueError(
        "Unexpected total number of tiny values."
    )


print(
    "\nPASS: All 1,691 tiny values occur only "
    "in CDC zero-permitted fields."
)


# ============================================================
# 8. RESTORE TRUE ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col in ZERO_ALLOWED_COLUMNS:

    mask = df[col].eq(
        TINY_VALUE
    )

    count = int(
        mask.sum()
    )

    print(
        f"{col:<10}: {count:,} tiny values -> 0"
    )

    df.loc[
        mask,
        col
    ] = 0.0


# ============================================================
# 9. VERIFY ZERO COUNTS
# ============================================================

print("\n--- RESTORED ZERO COUNTS ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_zero = int(
        df[col]
        .eq(0)
        .sum()
    )

    print(
        f"{col:<10}: {actual_zero:,}"
    )

    if actual_zero != expected_count:

        raise ValueError(
            f"{col}: zero restoration count differs."
        )


print(
    "PASS: All 1,691 zeros restored correctly."
)


# ============================================================
# 10. CHECK CODE FIELDS FOR REAL DECIMALS
# ============================================================

#
# Do NOT include the product measurements here.
#
# SMD100NI legitimately contains fractions.
#

print("\n--- QUESTIONNAIRE FRACTION CHECK ---")


for col in ["SEQN"] + CODE_COLUMNS:

    values = (
        df[col]
        .dropna()
    )

    if values.empty:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional.any():

        print(
            f"Unexpected real decimal values in {col}:"
        )

        print(
            values[
                fractional
            ].head(20)
        )

        raise ValueError(
            f"STOP: {col} contains genuine decimals. "
            "It will NOT be converted."
        )


print(
    "PASS: Questionnaire/code fields contain "
    "whole-number values only."
)


# ============================================================
# 11. CONFIRM GENUINE DECIMAL MEASUREMENT
# ============================================================

nicotine = df["SMD100NI"]


print("\n--- SMD100NI DECIMAL FIDELITY ---")

print(
    "Valid:",
    int(nicotine.notna().sum())
)

print(
    "Missing:",
    int(nicotine.isna().sum())
)

print(
    "Minimum:",
    nicotine.min()
)

print(
    "Maximum:",
    nicotine.max()
)


if (
    int(nicotine.notna().sum()) == 954
    and
    int(nicotine.isna().sum()) == 4087
    and
    np.isclose(
        nicotine.min(),
        0.1
    )
    and
    np.isclose(
        nicotine.max(),
        1.8
    )
):

    print(
        "PASS: SMD100NI matches CDC and will remain untouched."
    )

else:

    raise ValueError(
        "SMD100NI range/count differs from CDC."
    )


# ============================================================
# 12. CLEAN SEQN
# ============================================================

df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 13. CLEAN ONLY QUESTIONNAIRE CODE FIELDS
# ============================================================

#
# 1.0   -> 1
# 20.0  -> 20
# 999.0 -> 999
#
# Measurements and strings are NOT included here.
#

for col in CODE_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Questionnaire codes/counts formatted "
    "without unnecessary .0."
)


# ============================================================
# 14. CHARACTER FIDELITY - SMDUPCA
# ============================================================

upc = df["SMDUPCA"].astype(str)

upc_nonblank = (
    upc != ""
)


print("\n--- SMDUPCA ---")

print(
    "Recorded:",
    int(upc_nonblank.sum())
)

print(
    "Blank:",
    int((~upc_nonblank).sum())
)


if (
    int(upc_nonblank.sum()) != 884
    or
    int((~upc_nonblank).sum()) != 4157
):

    raise ValueError(
        "SMDUPCA recorded/blank counts differ from CDC."
    )


#
# Protect leading zeros.
#

leading_zero_count = int(
    upc[
        upc_nonblank
    ]
    .str.startswith("0")
    .sum()
)


print(
    "Recorded values beginning with 0:",
    leading_zero_count
)


#
# All nonblank values in this source occupy
# the 12-character UPC field, including
# special '* NO MATCH *' values.
#

bad_length = int(
    (
        upc[
            upc_nonblank
        ]
        .str.len()
        != 12
    )
    .sum()
)


print(
    "Nonblank values not 12 characters:",
    bad_length
)


if bad_length != 0:

    raise ValueError(
        "Unexpected SMDUPCA length."
    )


print(
    "PASS: SMDUPCA remains character data."
)


# ============================================================
# 15. CHARACTER FIDELITY - SMD100BR
# ============================================================

brand = (
    df["SMD100BR"]
    .astype(str)
)

brand_nonblank = (
    brand != ""
)


print("\n--- SMD100BR ---")

print(
    "Recorded:",
    int(brand_nonblank.sum())
)

print(
    "Blank:",
    int((~brand_nonblank).sum())
)


if (
    int(brand_nonblank.sum()) != 1103
    or
    int((~brand_nonblank).sum()) != 3938
):

    raise ValueError(
        "SMD100BR count differs from CDC."
    )


print(
    "PASS: Cigarette brand strings preserved."
)


# ============================================================
# 16. CDC CHECK - SMQ020
# ============================================================

expected = {
    1: 2497,
    2: 2538,
    7: 1,
    9: 4
}

actual = (
    df["SMQ020"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != expected
    or
    int(df["SMQ020"].isna().sum()) != 1
):

    raise ValueError(
        "SMQ020 differs from CDC."
    )


print(
    "\nPASS: SMQ020 matches CDC."
)


# ============================================================
# 17. CDC CHECK - SMD030
# ============================================================

s = df["SMD030"]


if not (
    int(s.between(5, 70).sum()) == 2322
    and
    int(s.eq(0).sum()) == 165
    and
    int(s.eq(999).sum()) == 10
    and
    int(s.isna().sum()) == 2544
):

    raise ValueError(
        "SMD030 differs from CDC."
    )


print(
    "PASS: SMD030 matches CDC."
)


# ============================================================
# 18. CDC CHECK - SMD641
# ============================================================

s = df["SMD641"]


if not (
    int(s.between(0, 30).sum()) == 1131
    and
    int(s.eq(99).sum()) == 1
    and
    int(s.isna().sum()) == 3909
):

    raise ValueError(
        "SMD641 differs from CDC."
    )


print(
    "PASS: SMD641 matches CDC."
)


# ============================================================
# 19. CDC CHECK - FILTER / MENTHOL
# ============================================================

if (
    df["SMD100FL"]
    .value_counts()
    .sort_index()
    .to_dict()
    !=
    {
        0: 27,
        1: 1076
    }
):

    raise ValueError(
        "SMD100FL differs from CDC."
    )


if int(df["SMD100FL"].isna().sum()) != 3938:

    raise ValueError(
        "SMD100FL missing count differs."
    )


print(
    "PASS: SMD100FL matches CDC."
)


if (
    df["SMD100MN"]
    .value_counts()
    .sort_index()
    .to_dict()
    !=
    {
        0: 732,
        1: 371
    }
):

    raise ValueError(
        "SMD100MN differs from CDC."
    )


if int(df["SMD100MN"].isna().sum()) != 3938:

    raise ValueError(
        "SMD100MN missing count differs."
    )


print(
    "PASS: SMD100MN matches CDC."
)


# ============================================================
# 20. CDC CHECK - OTHER ZERO-CODED AGE VARIABLES
# ============================================================

ZERO_AGE_CHECKS = {

    "SMD130": {
        "range": (6, 71),
        "range_count": 237,
        "zero_count": 215,
        "dk": 2,
        "missing": 4587
    },

    "SMD160": {
        "range": (6, 75),
        "range_count": 272,
        "zero_count": 420,
        "dk": 4,
        "missing": 4345
    },

    "SMD190": {
        "range": (7, 60),
        "range_count": 155,
        "zero_count": 38,
        "dk": 1,
        "missing": 4847
    },

    "SMD220": {
        "range": (4, 70),
        "range_count": 205,
        "zero_count": 82,
        "dk": 1,
        "missing": 4753
    }
}


print("\n--- ZERO-CODED AGE CHECKS ---")


for col, spec in ZERO_AGE_CHECKS.items():

    s = df[col]

    lo, hi = spec["range"]


    if not (
        int(
            s.between(
                lo,
                hi
            ).sum()
        )
        ==
        spec["range_count"]

        and

        int(
            s.eq(0).sum()
        )
        ==
        spec["zero_count"]

        and

        int(
            s.eq(999).sum()
        )
        ==
        spec["dk"]

        and

        int(
            s.isna().sum()
        )
        ==
        spec["missing"]
    ):

        raise ValueError(
            f"{col} differs from CDC."
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 21. PRODUCT MEASUREMENT CHECKS
# ============================================================

print("\n--- PRODUCT MEASUREMENT CHECKS ---")


MEASUREMENT_CHECKS = {

    "SMD100TR": {
        "valid": 955,
        "missing": 4086,
        "min": 1,
        "max": 25
    },

    "SMD100NI": {
        "valid": 954,
        "missing": 4087,
        "min": 0.1,
        "max": 1.8
    },

    "SMD100CO": {
        "valid": 954,
        "missing": 4087,
        "min": 1,
        "max": 20
    }
}


for col, spec in MEASUREMENT_CHECKS.items():

    s = df[col]


    if not (
        int(s.notna().sum())
        ==
        spec["valid"]

        and

        int(s.isna().sum())
        ==
        spec["missing"]

        and

        np.isclose(
            s.min(),
            spec["min"]
        )

        and

        np.isclose(
            s.max(),
            spec["max"]
        )
    ):

        raise ValueError(
            f"{col} differs from CDC."
        )


    print(
        f"PASS: {col} "
        f"range={s.min()}-{s.max()}"
    )


# ============================================================
# 22. BUILD EXPECTED SOURCE
# ============================================================

#
# Expected source =
#
# original XPT
#
# plus ONLY the 8 CDC-supported zero restorations.
#

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 23. NUMERIC FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


numeric_failures = []


for col in expected_df.columns:

    if col in CHARACTER_COLUMNS:
        continue


    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col == "SEQN" or col in CODE_COLUMNS:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )

    else:

        current_values = (
            df[col]
            .to_numpy()
        )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        numeric_failures.append(
            col
        )


if numeric_failures:

    raise ValueError(
        "Unexpected numeric changes in:\n"
        + str(numeric_failures)
    )


print(
    "PASS: Numeric values preserved exactly."
)


# ============================================================
# 24. CHARACTER FIDELITY BEFORE EXPORT
# ============================================================

for col in CHARACTER_COLUMNS:

    expected_values = (
        expected_df[col]
        .astype(str)
        .to_numpy()
    )

    current_values = (
        df[col]
        .astype(str)
        .to_numpy()
    )


    if not np.array_equal(
        expected_values,
        current_values
    ):

        raise ValueError(
            f"{col} character data changed."
        )


print(
    "PASS: SMDUPCA and SMD100BR preserved exactly."
)


# ============================================================
# 25. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\nRemaining tiny values:", remaining_tiny)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny zero artifacts still remain."
    )


print(
    "PASS: No tiny zero artifact remains."
)


# ============================================================
# 26. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format
# NO rounding
#
# SMD100TR / SMD100NI / SMD100CO remain as floats.
#
# SMDUPCA remains text, preserving leading zeros.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 27. RE-READ NUMERIC CSV
# ============================================================

csv_numeric = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 28. RE-READ CHARACTER COLUMNS AS TEXT
# ============================================================

csv_character = pd.read_csv(
    csv_file,
    usecols=CHARACTER_COLUMNS,
    dtype={
        "SMDUPCA": "string",
        "SMD100BR": "string"
    },
    keep_default_na=False
)


# ============================================================
# 29. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    if col in CHARACTER_COLUMNS:
        continue


    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_numeric[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences in:\n"
        + str(csv_failures)
    )


print(
    "PASS: Every numeric value matches expected XPT."
)


# ============================================================
# 30. CHARACTER CSV FIDELITY
# ============================================================

print("\n--- CHARACTER CSV FIDELITY ---")


for col in CHARACTER_COLUMNS:

    expected_values = (
        expected_df[col]
        .astype(str)
        .to_numpy()
    )

    exported_values = (
        csv_character[col]
        .astype(str)
        .to_numpy()
    )


    if not np.array_equal(
        expected_values,
        exported_values
    ):

        raise ValueError(
            f"{col} differs after CSV export."
        )


    print(
        f"PASS: {col} exact character match."
    )


# ============================================================
# 31. MISSING POSITION FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    if col in CHARACTER_COLUMNS:
        continue


    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_numeric[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            csv_missing
        )
    )


print("\nMissing-position mismatches:", missing_mismatches)


if missing_mismatches != 0:

    raise ValueError(
        "Numeric missing positions changed."
    )


print(
    "PASS: Missing numeric values preserved exactly."
)


# ============================================================
# 32. RAW CSV TOKEN CHECK
# ============================================================

#
# IMPORTANT:
#
# Do NOT globally reject ".0".
#
# SMD100TR / SMD100NI / SMD100CO are measurements
# and a source value such as 10.0 is allowed to stay 10.0.
#
# Only identifiers and questionnaire codes are required
# not to have unnecessary .0.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


bad_code_dot_zero = 0


for row in raw_rows:

    for col in ["SEQN"] + CODE_COLUMNS:

        value = row[col]

        if value.endswith(".0"):

            bad_code_dot_zero += 1


print(
    "Identifier/code values ending .0:",
    bad_code_dot_zero
)


if bad_code_dot_zero != 0:

    raise ValueError(
        "Unnecessary .0 remains in code fields."
    )


print(
    "PASS: SEQN/questionnaire codes have no .0."
)


# ============================================================
# 33. RAW MEASUREMENT TOKEN FIDELITY
# ============================================================

#
# Compare CSV text directly with the XPT float
# representation.
#

measurement_mismatches = {
    col: 0
    for col in MEASUREMENT_COLUMNS
}


for i, row in enumerate(raw_rows):

    for col in MEASUREMENT_COLUMNS:

        xpt_value = expected_df.at[
            i,
            col
        ]


        expected_text = (
            ""
            if pd.isna(xpt_value)
            else str(float(xpt_value))
        )


        if row[col] != expected_text:

            measurement_mismatches[col] += 1


print(
    "Measurement raw-text mismatches:",
    measurement_mismatches
)


if any(
    measurement_mismatches.values()
):

    raise ValueError(
        "Measurement formatting changed."
    )


print(
    "PASS: Product measurements appear AS IS in CSV."
)


# ============================================================
# 34. UPC RAW FIDELITY
# ============================================================

upc_mismatch = 0


for i, row in enumerate(raw_rows):

    expected_upc = str(
        expected_df.at[
            i,
            "SMDUPCA"
        ]
    )

    if row["SMDUPCA"] != expected_upc:

        upc_mismatch += 1


print(
    "SMDUPCA raw-text mismatches:",
    upc_mismatch
)


if upc_mismatch != 0:

    raise ValueError(
        "SMDUPCA changed during CSV export."
    )


print(
    "PASS: UPC leading zeros/text preserved exactly."
)


# ============================================================
# 35. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


print("\n" + "=" * 78)
print("SMQ_C CONVERSION COMPLETE")
print("=" * 78)

NHANES 2003-2004 SMQ_C XPT -> CSV
Rows: 5,041
Columns: 42
PASS: SMQ_C structure = 5,041 rows x 42 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny-value occurrences: 1,691

Tiny values by variable:
SMD100MN    732
SMD160      420
SMD130      215
SMD030      165
SMD220       82
SMD190       38
SMD100FL     27
SMD641       12

PASS: All 1,691 tiny values occur only in CDC zero-permitted fields.

--- ZERO RESTORATION ---
SMD030    : 165 tiny values -> 0
SMD641    : 12 tiny values -> 0
SMD100FL  : 27 tiny values -> 0
SMD100MN  : 732 tiny values -> 0
SMD130    : 215 tiny values -> 0
SMD160    : 420 tiny values -> 0
SMD190    : 38 tiny values -> 0
SMD220    : 82 tiny values -> 0

--- RESTORED ZERO COUNTS ---
SMD030    : 165
SMD641    : 12
SMD100FL  : 27
SMD100MN  : 732
SMD130    : 215
SMD160    : 420
SMD190    : 38
SMD220    : 82
PASS: All 1,691 zeros restored correctly.

--- QUESTIONNAIRE FRACTION CHECK ---
PASS: Questionnaire/code fields contain whole-number values only